In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-06 09:23:36--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:800:b:20a5:b140:21, 2600:9000:234c:fe00:b:20a5:b140:21, 2600:9000:234c:d000:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0% 3.98M 17s
    50K .......... .......... .......... .......... ..........  0%  453M 9s
   100K .......... .......... .......... .......... ..........  0% 8.03M 9s
   150K .......... .......... .......... .......... ..........  0% 7.74M 9s
   200K .......... .......... .......... .......... ..........  0% 45.8M 7s
   250K .......... .......... .......... .......... .........

In [3]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [4]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [5]:
df_yellow.repartition(4).write.parquet('data/pq/yellow/2025/11/')

In [7]:
from pyspark.sql import functions as F

In [8]:
trip_count = df_yellow \
    .withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15') \
    .count()

In [9]:
print(f"Number of trips on November 15, 2025: {trip_count}")

Number of trips on November 15, 2025: 162604


In [10]:
df_with_duration = df_yellow.withColumn(
    'trip_duration_hrs', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

In [11]:
max_duration = df_with_duration.select(F.max('trip_duration_hrs')).collect()[0][0]

In [12]:
print(f"The longest trip duration is: {max_duration:.2f} hours")

The longest trip duration is: 90.65 hours


In [14]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv > taxi_zone_lookup.csv

--2026-03-06 16:57:30--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:5c00:b:20a5:b140:21, 2600:9000:234c:4e00:b:20a5:b140:21, 2600:9000:234c:2000:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:5c00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv.1'

     0K .......... ..                                         100%  700M=0s

2026-03-06 16:57:30 (700 MB/s) - 'taxi_zone_lookup.csv.1' saved [12331/12331]



In [23]:
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

In [24]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [25]:
pickup_counts = df_yellow.groupBy('PULocationID').count()

In [26]:
df_result = pickup_counts.join(df_zones, F.col("PULocationID") == F.col("LocationID"))

In [27]:
df_result.orderBy(F.col("count").asc()).limit(5).show()

+------------+-----+----------+-------------+--------------------+------------+
|PULocationID|count|LocationID|      Borough|                Zone|service_zone|
+------------+-----+----------+-------------+--------------------+------------+
|         105|    1|       105|    Manhattan|Governor's Island...| Yellow Zone|
|          84|    1|        84|Staten Island|Eltingville/Annad...|   Boro Zone|
|           5|    1|         5|Staten Island|       Arden Heights|   Boro Zone|
|         187|    3|       187|Staten Island|       Port Richmond|   Boro Zone|
|         111|    4|       111|     Brooklyn| Green-Wood Cemetery|   Boro Zone|
+------------+-----+----------+-------------+--------------------+------------+

